# 02 - Identificação de Fundos REAG

Identifica fundos administrados/geridos pela REAG usando dados de cadastro da CVM.

**REAG**: Administradora investigada por fraude junto ao Banco Master

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from pathlib import Path
from src.processors.data_processor import DataProcessor
from config.settings import Config

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

## Carregar Cadastro de Fundos

In [2]:
config = Config()
processor = DataProcessor(config)

# Ler cadastro mais recente
# CVM files are named: cad_fi.csv (current) or cad_fi_hist.zip (historical)
cadastro_files = list(config.RAW_DATA_DIR.glob('cad_fi*.csv'))

# Also check in cadastro subdirectory if it exists
cadastro_dir = config.RAW_DATA_DIR / 'cadastro'
if cadastro_dir.exists():
    cadastro_files.extend(list(cadastro_dir.glob('cad_fi*.csv')))

# Sort by modification time (most recent first)
cadastro_files = sorted(cadastro_files, key=lambda p: p.stat().st_mtime, reverse=True)
latest_cadastro = cadastro_files[0] if cadastro_files else None

if latest_cadastro:
    print(f"📂 Lendo: {latest_cadastro}")
    df_cadastro = processor.read_cadastro(latest_cadastro)
    print(f"📊 Total de fundos no cadastro: {len(df_cadastro):,}")
else:
    print("⚠️ Nenhum arquivo de cadastro encontrado")
    print(f"   Procurado em: {config.RAW_DATA_DIR}")
    print(f"   Padrão: cad_fi*.csv")
    print("\n💡 Execute o notebook 01_data_collection.ipynb para baixar os dados")

📂 Lendo: /Users/pedrotodescan/Documents/code/REAG/notebooks/../data/raw/cad_fi.csv
📊 Total de fundos no cadastro: 46,824


/Users/pedrotodescan/Documents/code/REAG/notebooks/../src/processors/data_processor.py:114: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding=encoding, sep=sep)


## Buscar REAG/CBSF nos Administradores

In [3]:
# Buscar por nome contendo 'REAG' ou 'CBSF'
search_terms = ['REAG', 'CBSF', 'BANCO MASTER']

mask = df_cadastro['DENOM_SOCIAL'].str.contains('|'.join(search_terms), case=False, na=False)
if 'ADMIN' in df_cadastro.columns:
    mask |= df_cadastro['ADMIN'].str.contains('|'.join(search_terms), case=False, na=False)
if 'GESTOR' in df_cadastro.columns:
    mask |= df_cadastro['GESTOR'].str.contains('|'.join(search_terms), case=False, na=False)

df_reag_related = df_cadastro[mask].copy()

print(f"\n🎯 Fundos relacionados encontrados: {len(df_reag_related)}")
print("\n📋 Amostra:")
display(df_reag_related[['CNPJ_FUNDO', 'DENOM_SOCIAL', 'SIT']].head(20))


🎯 Fundos relacionados encontrados: 318

📋 Amostra:


,CNPJ_FUNDO,DENOM_SOCIAL,SIT
6287,15.637.811/0001-75,AWM EQUITY HEDGE FUNDO DE INVESTIMENTO EM COTA...,CANCELADA
8437,11.741.429/0001-56,QUASAR TROPOS FUNDO DE INVESTIMENTO DE AÇÕES,CANCELADA
9299,16.858.916/0001-17,REAG HY DINAMICO FUNDO DE INVESTIMENTO RENDA F...,CANCELADA
9559,20.725.837/0001-05,NOVOTEMPO FUNDO DE INVESTIMENTO MULTIMERCADO C...,CANCELADA
9895,21.525.857/0001-03,REAG YEM FUNDO DE INVESTIMENTO MULTIMERCADO - ...,CANCELADA
9916,21.556.651/0001-32,REAG JUVIA FUNDO DE INVESTIMENTO MULTIMERCADO ...,CANCELADA
9936,21.596.695/0001-96,HELVETIA FUNDO DE INVESTIMENTO MULTIMERCADO CR...,LIQUIDAÇÃO
10331,22.614.205/0001-08,REAG CASH PLUS FUNDO DE INVESTIMENTO MULTIMERC...,CANCELADA
11272,25.044.388/0001-53,REAG SISTEMÁTICO FUNDO DE INVESTIMENTO MULTIME...,CANCELADA
11347,25.333.736/0001-02,REAG OLIMPIA FUNDO DE INVESTIMENTO MULTIMERCAD...,CANCELADA


## Identificar CNPJs de Administradores/Gestores REAG

In [4]:
# Analisar administradores únicos
if 'CNPJ_ADMIN' in df_cadastro.columns and 'ADMIN' in df_cadastro.columns:
    admin_counts = df_cadastro.groupby(['CNPJ_ADMIN', 'ADMIN']).size().reset_index(name='NUM_FUNDOS')
    admin_counts = admin_counts.sort_values('NUM_FUNDOS', ascending=False)
    
    # Filtrar REAG
    reag_admins = admin_counts[
        admin_counts['ADMIN'].str.contains('|'.join(search_terms), case=False, na=False)
    ]
    
    print("\n🏢 Administradores REAG/relacionados:")
    display(reag_admins)
    
    # Salvar CNPJs para uso posterior
    reag_admin_cnpjs = reag_admins['CNPJ_ADMIN'].tolist()
    print(f"\n📝 CNPJs de administradores REAG: {len(reag_admin_cnpjs)}")
    print(reag_admin_cnpjs)
else:
    print("⚠️ Colunas de administrador não encontradas")
    reag_admin_cnpjs = []


🏢 Administradores REAG/relacionados:


,CNPJ_ADMIN,ADMIN,NUM_FUNDOS
161,34.829.992/0001-86,REAG TRUST DISTRIBUIDORA DE TITULOS E VALORES ...,167
120,23.863.529/0001-34,REAG TRUST ADMINISTRADORA DE RECURSOS LTDA.,64



📝 CNPJs de administradores REAG: 2
['34.829.992/0001-86', '23.863.529/0001-34']


## Obter Lista Completa de Fundos REAG

In [5]:
# Filtrar fundos por CNPJ do administrador
if reag_admin_cnpjs:
    df_reag_funds = processor.filter_by_administrador(df_cadastro, reag_admin_cnpjs)
    
    print(f"\n💼 Total de fundos administrados pela REAG: {len(df_reag_funds)}")
    
    # Análise por situação
    if 'SIT' in df_reag_funds.columns:
        print("\n📊 Distribuição por situação:")
        display(df_reag_funds['SIT'].value_counts())
    
    # Fundos ativos
    df_reag_active = df_reag_funds[df_reag_funds['SIT'] == 'EM FUNCIONAMENTO NORMAL'].copy()
    print(f"\n✅ Fundos em funcionamento normal: {len(df_reag_active)}")
    
    # Salvar lista de CNPJs
    reag_fund_cnpjs = df_reag_funds['CNPJ_FUNDO'].unique().tolist()
    
    # Exportar para CSV
    output_path = config.PROCESSED_DATA_DIR / 'reag_fund_list.csv'
    df_reag_funds[['CNPJ_FUNDO', 'DENOM_SOCIAL', 'SIT', 'CNPJ_ADMIN']].to_csv(
        output_path, 
        index=False
    )
    print(f"\n💾 Lista salva em: {output_path}")
else:
    print("⚠️ Nenhum CNPJ de administrador REAG identificado")
    reag_fund_cnpjs = []


💼 Total de fundos administrados pela REAG: 231

📊 Distribuição por situação:


SIT
CANCELADA     216
LIQUIDAÇÃO     11
EM ANÁLISE      4
Name: count, dtype: int64


✅ Fundos em funcionamento normal: 0

💾 Lista salva em: /Users/pedrotodescan/Documents/code/REAG/notebooks/../data/processed/reag_fund_list.csv


## Resumo

In [6]:
print("\n" + "="*60)
print("📊 RESUMO DA IDENTIFICAÇÃO")
print("="*60)
print(f"Administradores REAG identificados: {len(reag_admin_cnpjs)}")
print(f"Fundos REAG identificados: {len(reag_fund_cnpjs)}")
print(f"Fundos ativos: {len(df_reag_active) if 'df_reag_active' in locals() else 0}")
print(f"\n📂 Lista exportada: {config.PROCESSED_DATA_DIR / 'reag_fund_list.csv'}")
print("\n✅ Identificação concluída! Próximo passo: 03_flow_analysis.ipynb")


📊 RESUMO DA IDENTIFICAÇÃO
Administradores REAG identificados: 2
Fundos REAG identificados: 215
Fundos ativos: 0

📂 Lista exportada: /Users/pedrotodescan/Documents/code/REAG/notebooks/../data/processed/reag_fund_list.csv

✅ Identificação concluída! Próximo passo: 03_flow_analysis.ipynb
